In [ ]:
# Original Hero1 standalone GPU component diagnostic, not a trained candidate.
import os, sys, json, hashlib, importlib.util
from pathlib import Path
import torch, triton
repo=Path(os.environ["SOUP_REPO"])
results=Path(os.environ["SOUP_RESULTS"])
results.mkdir(parents=True, exist_ok=True)
assert torch.cuda.device_count()==1, "Requires single allocated challenge GPU"
assert torch.cuda.get_device_name(0)=="Tesla T4"
src="\"\"\"Hero1 independent one-pass FP16-logit vocabulary statistics.\n\nAlgorithmic inspiration: public online-softmax and fused CE/entropy/argmax designs;\nsee the project research references. This is an original implementation, not\ncopied from a competitor kernel or the challenge starter. This module is an\nUNINTEGRATED COMPONENT PROTOTYPE, not a qualifying training solution.\n\nFor each row we maintain stable (max, sum_exp, centered_weighted_sum),\nand the first-index argmax. The centered accumulation avoids subtracting\nlarge nearly equal terms when logits have common offsets. The operation\nreads the ORIGINAL FP16 logits and respects ignore_index=-100.\n\nNo weight or optimizer changes, no changes to training data or metrics.\n\"\"\"\nimport torch\nimport triton\nimport triton.language as tl\n\n\n@triton.jit\ndef hero1_online_vocab_stats(\n    SCORES, TARGETS,\n    CE_OUT, ENTROPY_OUT, ARGMAX_OUT, MAX_OUT, LOGNORM_OUT,\n    VOCAB: tl.constexpr, BLOCK: tl.constexpr,\n):\n    row = tl.program_id(0)\n    lane = tl.arange(0, BLOCK)\n    mx = tl.full((), -float(\"inf\"), tl.float32)\n    total = tl.full((), 0.0, tl.float32)\n    centered = tl.full((), 0.0, tl.float32)\n    winner = tl.full((), 2147483647, tl.int32)\n\n    # One CTA owns one complete row. The scan itself performs useful\n    # statistics on all logits, including ignored targets needed for entropy.\n    for tile in range(tl.cdiv(VOCAB, BLOCK)):\n        idx = tile * BLOCK + lane\n        inside = idx < VOCAB\n        x = tl.load(SCORES + row * VOCAB + idx, inside, other=-float(\"inf\")).to(tl.float32)\n        batch_max = tl.max(x, 0)\n        new_mx = tl.maximum(mx, batch_max)\n        has_old = mx != -float(\"inf\")\n        delta_old = tl.where(has_old, mx - new_mx, 0.0)\n        factor = tl.where(has_old, tl.exp(delta_old), 0.0)\n        expval = tl.where(inside, tl.exp(x - new_mx), 0.0)\n        part_total = tl.sum(expval, 0)\n        part_centered = tl.sum(tl.where(inside, expval * (x - new_mx), 0.0), 0)\n        new_centered = factor * (centered + delta_old * total) + part_centered\n        new_total = factor * total + part_total\n\n        this_winner = tl.min(tl.where(inside & (x == batch_max), idx, 2147483647), 0)\n        winner = tl.where(batch_max > mx, this_winner,\n                          tl.where(batch_max == mx, tl.minimum(winner, this_winner), winner))\n        mx = new_mx\n        total = new_total\n        centered = new_centered\n\n    log_norm = tl.log(total)\n    entropy = log_norm - centered / total\n    label = tl.load(TARGETS + row)\n    supervised = label != -100\n    safe_label = tl.where(supervised, label, 0)\n    target_val = tl.load(SCORES + row * VOCAB + safe_label, supervised, other=0.).to(tl.float32)\n    ce = tl.where(supervised, -((target_val - mx) - log_norm), 0.0)\n    tl.store(CE_OUT + row, ce)\n    tl.store(ENTROPY_OUT + row, entropy)\n    tl.store(ARGMAX_OUT + row, winner)\n    tl.store(MAX_OUT + row, mx)\n    tl.store(LOGNORM_OUT + row, log_norm)\n\n\ndef online_vocab_stats(logits, labels, *, block=2048):\n    if not (isinstance(logits, torch.Tensor) and logits.is_cuda and\n            logits.dtype == torch.float16 and logits.ndim == 2 and\n            logits.is_contiguous() and isinstance(labels, torch.Tensor) and\n            labels.is_cuda and labels.dtype == torch.long and\n            labels.ndim == 1 and labels.numel() == logits.shape[0] and\n            logits.shape[1] > 0 and block in (1024, 2048, 4096)):\n        raise ValueError(\"Hero1 diagnostic supports contiguous CUDA FP16 [rows,vocab] and int64 targets only\")\n    rows, vocab = logits.shape\n    ce = torch.empty(rows, device=logits.device, dtype=torch.float32)\n    h = torch.empty_like(ce)\n    argmax = torch.empty(rows, device=logits.device, dtype=torch.long)\n    mx = torch.empty_like(ce)\n    lognorm = torch.empty_like(ce)\n    hero1_online_vocab_stats[(rows,)](\n        logits, labels, ce, h, argmax, mx, lognorm,\n        VOCAB=vocab, BLOCK=block, num_warps=4, enable_fp_fusion=False,\n    )\n    return ce, h, argmax, mx, lognorm\n"
modpath=repo/"solution"/"hero1_online_vocab_stats.py"
assert modpath.parent.is_dir()
modpath.write_text(src, encoding="utf-8")
compile(src, str(modpath), "exec")
spec=importlib.util.spec_from_file_location("hero1_online_vocab_stats",modpath)
online=importlib.util.module_from_spec(spec)
spec.loader.exec_module(online)
print("original_module_sha256",hashlib.sha256(src.encode()).hexdigest())
print("GPU",torch.cuda.get_device_name(0),"Triton",triton.__version__)


In [ ]:
# Test independently derived one-pass centered online softmax recurrence.
import torch.nn.functional as F
torch.manual_seed(20261031)
torch.cuda.manual_seed_all(20261031)
checks=[]
for shape, scale, add, peak in [
    ((8, 4096), 1.0, 0.0, 0.0),
    ((32, 151936), 0.8, 0.0, 0.0),
    ((128, 151936), 2.0, 10.0, 0.0),
    ((32, 151936), 1.0, 0.0, 35.0),
]:
    rows,vocab=shape
    x=(torch.randn(shape,device="cuda",dtype=torch.float32)*scale+add).half().contiguous()
    if peak:
        x[:,0]=x[:,0]+peak
        x[:,11]=x[:,11]+peak # exact-tie adversarial peaks
    labels=torch.randint(0,vocab,(rows,),device="cuda",dtype=torch.long)
    labels[::7]=-100
    ce,h,p,m,logn=online.online_vocab_stats(x,labels,block=2048)
    native=F.log_softmax(x.float(),dim=-1)
    ce_ref=F.nll_loss(native,labels,reduction="none",ignore_index=-100)
    h_ref=-(native.exp()*native).sum(-1)
    p_ref=x.argmax(-1)
    ce_error=(ce-ce_ref).abs().max().item()
    h_error=(h-h_ref).abs().max().item()
    assert torch.equal(p,p_ref),f"argmax mismatch for {shape}"
    assert torch.all(ce[labels==-100]==0),"ignored CE not zero"
    assert torch.isfinite(ce).all() and torch.isfinite(h).all()
    assert ce_error < 0.0025, f"CE {ce_error} for {shape}"
    assert h_error < 0.005, f"Entropy {h_error} for {shape}"
    checks.append(dict(rows=rows,vocab=vocab,scale=scale,shift=add,peak=peak,ce_max_abs_error=ce_error,entropy_max_abs_error=h_error,first_argmax_tie_policy="first index",passed=True))
    print("PASS",checks[-1])


In [ ]:
# One-block-size actual T4 timing only: no warm-cache competition claim.
# Output is a microbenchmark, NOT whole soup train nor proof of required 5% share.
import torch
rows,vocab=128,151936
x=torch.randn((rows,vocab),device="cuda",dtype=torch.float16).contiguous()
y=torch.randint(vocab,(rows,),device="cuda",dtype=torch.long)
for _ in range(3):
    online.online_vocab_stats(x,y,block=2048)
torch.cuda.synchronize()
start=torch.cuda.Event(enable_timing=True)
stop=torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(10):
    online.online_vocab_stats(x,y,block=2048)
stop.record(); torch.cuda.synchronize()
elapsed_ms=start.elapsed_time(stop)/10
report={
    "scope":"component_only_unintegrated",
    "eligible_for_competition":False,
    "not_a_training_measurement":True,
    "original_kernel":"hero1_online_vocab_stats",
    "size":[rows,vocab],
    "block":2048,
    "ms_per_launch_including_allocs":elapsed_ms,
    "test_cases":checks,
    "gpu":torch.cuda.get_device_name(0),
    "triton":triton.__version__,
}
results.joinpath("hero1_online_vocab_component.json").write_text(json.dumps(report,indent=2))
print("COMPONENT RESULT",json.dumps(report,indent=2))
